In [ ]:
import torch
import numpy as np
import logging
import os
from dotenv import load_dotenv

from libs.client import Client, SplitType
from libs.model import ContextAwareActor, ContextAwareCritic, Recommender, RecommenderTrainer

load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# ===================== PATHS =====================
USER_ID = "user_55239"
SELECTED_USER_COMPLETE_PATH = f"/mnt/ssd/Carrera/5th_Year/X_SEMESTER/PFC_3/Dataset/processed_users/{USER_ID}_processed.csv"
EMBEDDINGS_CSV_PATH = "/mnt/ssd/Carrera/Datasets/Music4all-Onion/music_4_all_compress_64.csv"
CACHE_PATH = "cache_client.json"
SAVE_PATH = "modelo_recomendacion"

API_URL = os.getenv("API_URL")
EMBEDDING_URL = f"{API_URL}/info"

# ===================== DATA =====================
SPLIT_RATIOS = (0.70, 0.15, 0.15)       # (train, test, validation)
CLIENT_BATCH_SIZE = 32

# ===================== MODEL =====================
EMBEDDING_DIM = 64
HIDDEN_DIM = 256
STATE_DIM = HIDDEN_DIM + 32             # history_rep + context_rep
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ===================== REWARD =====================
REWARD_ALPHA = 0.6                       # weight for interaction_count
REWARD_BETA = 0.4                        # weight for interaction_ratio

# ===================== ACTOR ARCHITECTURE =====================
ACTOR_DROPOUT = 0.1
ACTOR_INIT_GAIN = 0.1
NUM_ATTENTION_HEADS = 4

# ===================== CRITIC ARCHITECTURE =====================
CRITIC_DROPOUT = 0.1
CRITIC_INIT_GAIN = 0.01
CRITIC_OUTPUT_INIT_RANGE = 0.003
CRITIC_Q_CLAMP = 10.0

# ===================== DDPG TRAINING =====================
ACTOR_LR = 1e-6
CRITIC_LR = 1e-3
ACTOR_WEIGHT_DECAY = 1e-4
CRITIC_WEIGHT_DECAY = 1e-6
GAMMA = 0.9                             # discount factor
TAU = 0.005                             # soft target update rate
BATCH_SIZE = 64                         # replay buffer sample size
STATE_SIZE = 10                          # history window length
MEMORY_CAPACITY = 10_000                 # replay buffer capacity
TARGET_UPDATE_FREQ = 100                 # steps between target network updates
GRAD_CLIP_NORM = 1.0                    # max gradient norm
ACTOR_NOISE_SCALE = 0.1                 # noise scale in actor update
ENTROPY_COEFF = 0.01                    # entropy regularization weight

# ===================== EXPLORATION =====================
NUM_EPOCHS = 50
EPSILON_START = 0.9
EPSILON_END = 0.1
EPSILON_DECAY = 0.995                    # per-epoch multiplicative decay
EXPLORATION_NOISE = 0.2                  # gaussian noise scale during training
MAX_TRAIN_STEPS = 1000                   # steps per training epoch
MAX_EVAL_STEPS = 500                     # steps per evaluation

# ===================== EVALUATION =====================
EVAL_FREQ = 1                            # evaluate every N epochs

print(f"Device: {DEVICE}")
print(f"Epsilon after {NUM_EPOCHS} epochs: {EPSILON_START * EPSILON_DECAY**NUM_EPOCHS:.4f}")

In [ ]:
def calcular_recompensa_normalizada(interaction_count: int, interaction_ratio: float) -> float:
    """Recompensa acotada entre 0 y 1 para estabilidad."""
    count_factor = np.log1p(interaction_count) / np.log1p(10)
    recompensa = REWARD_ALPHA * count_factor + REWARD_BETA * interaction_ratio
    return float(np.clip(recompensa, 0.0, 1.0))

def ejemplo_get_embedding(track_id: str) -> torch.Tensor:
    import requests
    response = requests.get(f"{EMBEDDING_URL}/{track_id}?info_type=embedding")
    if response.status_code == 200:
        embedding = response.json().get("data", {}).get("embedding", [])
        return torch.tensor(embedding, dtype=torch.float32)
    else:
        logging.error(f"Error al obtener embedding para track_id {track_id}: {response.status_code}")
        return torch.zeros(EMBEDDING_DIM, dtype=torch.float32)

In [ ]:
client = Client(
    path=SELECTED_USER_COMPLETE_PATH,
    recompensa_func=calcular_recompensa_normalizada,
    get_embedding_func=ejemplo_get_embedding,
    batch_size=CLIENT_BATCH_SIZE,
    split_ratios=SPLIT_RATIOS,
    cache_path=CACHE_PATH,
    embeddings_path=EMBEDDINGS_CSV_PATH,
)

In [ ]:
actor = ContextAwareActor(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=ACTOR_DROPOUT,
    init_gain=ACTOR_INIT_GAIN,
    num_heads=NUM_ATTENTION_HEADS,
).to(DEVICE)

critic = ContextAwareCritic(
    state_dim=STATE_DIM,
    action_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=CRITIC_DROPOUT,
    init_gain=CRITIC_INIT_GAIN,
    output_init_range=CRITIC_OUTPUT_INIT_RANGE,
    q_clamp=CRITIC_Q_CLAMP,
).to(DEVICE)

recommender = Recommender(client=client)

recommender_trainer = RecommenderTrainer(
    actor=actor,
    critic=critic,
    client=client,
    recommender=recommender,
    gamma=GAMMA,
    tau=TAU,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    batch_size=BATCH_SIZE,
    state_size=STATE_SIZE,
    device=DEVICE,
    target_update_freq=TARGET_UPDATE_FREQ,
    actor_weight_decay=ACTOR_WEIGHT_DECAY,
    critic_weight_decay=CRITIC_WEIGHT_DECAY,
    memory_capacity=MEMORY_CAPACITY,
    grad_clip_norm=GRAD_CLIP_NORM,
    actor_noise_scale=ACTOR_NOISE_SCALE,
    entropy_coeff=ENTROPY_COEFF,
    max_train_steps=MAX_TRAIN_STEPS,
    max_eval_steps=MAX_EVAL_STEPS,
    exploration_noise=EXPLORATION_NOISE,
)

In [ ]:
history = recommender_trainer.train(
    num_epochs=NUM_EPOCHS,
    epsilon_start=EPSILON_START,
    epsilon_end=EPSILON_END,
    epsilon_decay=EPSILON_DECAY,
    eval_freq=EVAL_FREQ,
    save_path=SAVE_PATH,
)

In [ ]:
recommender_trainer.plot_individual_metrics()